# 4. Complex open-platform queries

Compose continuous and categorical FILTER clauses, combine them under AND / OR groups, and nest groups inside groups. Same `createSubQuery()` / `buildQuery()` / `runQuery()` API used throughout.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
open_hpds_session <- picsure::connect(
  platform = picsure::Platform$BDC_OPEN,
  token    = my_token
)

## Restrict the search to two studies

In [ ]:
two_studies <- picsure::facets(open_hpds_session)
picsure::addFacet(two_studies, "dataset_id", c("phs000810", "phs000007"))

In [ ]:
results <- picsure::searchDictionary(open_hpds_session, "age", facets = two_studies)
results

## Pick the two AGE variables to use

In [ ]:
age_immi_phs000810 <- results[results$display == "AGE_IMMI", ]
age_immi_phs000810

In [ ]:
age5_phs000007 <- results[results$display == "age5" & results$name == "phv00177938", ]
age5_phs000007

## Continuous FILTER (numeric range)

Pass `min` and `max` instead of `categories`.

In [ ]:
age5_phs000007_clause <- picsure::createSubQuery(
  age5_phs000007$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

picsure::runQuery(open_hpds_session, age5_phs000007_clause)

In [ ]:
age_immi_phs000810_clause <- picsure::createSubQuery(
  age_immi_phs000810$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

picsure::runQuery(open_hpds_session, age_immi_phs000810_clause)

## Combine with OR

Counts participants who match *either* clause.

In [ ]:
clause_group_or <- picsure::buildQuery(
  list(age_immi_phs000810_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$OR
)

picsure::runQuery(open_hpds_session, clause_group_or)

## Add a categorical FILTER on sex

In [ ]:
fhs_facet <- picsure::facets(open_hpds_session)
picsure::addFacet(fhs_facet, "dataset_id", "phs000007")

fhs_sex_results <- picsure::searchDictionary(open_hpds_session, "phv00253990", facets = fhs_facet)
fhs_sex_results

In [ ]:
fhs_sex_male_clause <- picsure::createSubQuery(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Male")
)

picsure::runQuery(open_hpds_session, fhs_sex_male_clause)

## Combine with AND

Males aged 30–40.

In [ ]:
fhs_male_and_30_to_40 <- picsure::buildQuery(
  list(fhs_sex_male_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

picsure::runQuery(open_hpds_session, fhs_male_and_30_to_40)

In [ ]:
fhs_sex_female_clause <- picsure::createSubQuery(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Female")
)

picsure::runQuery(open_hpds_session, fhs_sex_female_clause)

In [ ]:
fhs_female_and_30_to_40 <- picsure::buildQuery(
  list(fhs_sex_female_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

picsure::runQuery(open_hpds_session, fhs_female_and_30_to_40)

## Nested groups

`buildQuery()` accepts both leaf clauses *and* other groups. Nest groups inside groups to express arbitrary Boolean trees — here, OR-of-ANDs.

In [ ]:
fhs_female_ages_30_to_40_or_male_ages_30_to_40 <- picsure::buildQuery(
  list(fhs_female_and_30_to_40, fhs_male_and_30_to_40),
  operator = picsure::GroupOperator$OR
)

picsure::runQuery(open_hpds_session, fhs_female_ages_30_to_40_or_male_ages_30_to_40)